# EasyMagpieTTS thin server — single request

Send one TTS request to the thin HTTP server (`scripts/tts_server.py`) and play the result.

Start the server first (in the `emp` env):

```bash
./run_thin_server.sh /path/to/easymp_vllm_model 8091
```

The server streams audio as newline-delimited JSON (`application/x-ndjson`): one
`{"type": "audio", "sr", "pcm_b64"}` line per codec window, then a `{"type": "done", ...}`
line. We measure **time-to-first-audio (TTFA)** as the arrival of the first audio line.

In [ ]:
import base64
import json
import time

import numpy as np
import requests
import matplotlib.pyplot as plt
from IPython.display import Audio, display

SERVER_URL = "http://localhost:8091"

# Sanity check that the server is up and the engine finished loading.
print(requests.get(f"{SERVER_URL}/health", timeout=5).json())

In [ ]:
def synthesize(
    text: str,
    speaker_id: str | None = None,
    max_new_tokens: int = 1024,
    timeout: float = 300.0,
) -> tuple[np.ndarray, int]:
    """Send a streaming TTS request and collect the audio chunks."""
    payload = {"text": text, "stream": True, "max_new_tokens": max_new_tokens}
    if speaker_id:
        payload["speaker_id"] = speaker_id

    chunks: list[np.ndarray] = []
    sr = 22050
    t0 = time.perf_counter()
    t_first = None

    with requests.post(f"{SERVER_URL}/tts", json=payload, stream=True, timeout=timeout) as resp:
        resp.raise_for_status()
        for line in resp.iter_lines(decode_unicode=True):
            if not line:
                continue
            obj = json.loads(line)
            if obj["type"] == "audio":
                if t_first is None:
                    t_first = time.perf_counter()
                chunks.append(np.frombuffer(base64.b64decode(obj["pcm_b64"]), dtype=np.float32))
                sr = obj["sr"]
            elif obj["type"] == "error":
                raise RuntimeError(obj["message"])
            elif obj["type"] == "done":
                sr = obj.get("sr", sr)

    elapsed = time.perf_counter() - t0
    ttfa = (t_first - t0) if t_first is not None else elapsed
    audio = np.concatenate(chunks) if chunks else np.array([], dtype=np.float32)
    print(f"{len(chunks)} chunks | TTFA: {ttfa * 1000:.0f}ms | total: {elapsed:.3f}s")
    return audio, sr

In [ ]:
audio, sr = synthesize(
    text="Since then physicists have found that it is not reflection, but refraction by the raindrops which causes the rainbows.",
)
print(f"Got {len(audio)} samples — {len(audio) / sr:.2f}s @ {sr} Hz")

In [ ]:
t = np.arange(len(audio)) / sr

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t, audio, linewidth=0.3)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("EasyMagpieTTS waveform")
fig.tight_layout()
plt.show()

display(Audio(audio, rate=sr))